# ARC-AGI — nanoGPT Fine-tuning

Fine-tunes GPT-2 small on ARC task examples paired with LARC human descriptions.
The model learns to: (1) read training pairs, (2) state the transformation rule in
natural language, (3) predict the test output grid.

**Cell order:**
1. Check GPU → 2. Mount Drive → 3. Clone repos + install → 4. ARC data
→ 5. Generate dataset → 6. Set up nanoGPT → 7. Train → 8. Sample → 9. Download checkpoint

**Runtime:** A100 recommended.

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT_DIR = '/content/drive/MyDrive/arc-agi/nanogpt'
import os; os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

In [ ]:
# ── Cell 3: Clone repos + install dependencies ──────────────────────────
import os, subprocess, sys

def run(cmd): subprocess.run(cmd, check=True)

# nanoGPT
if not os.path.exists('/content/nanoGPT'):
    run(['git', 'clone', 'https://github.com/karpathy/nanoGPT', '/content/nanoGPT'])
else:
    run(['git', '-C', '/content/nanoGPT', 'pull'])

# arc-agi-solver
if not os.path.exists('/content/arc-agi-solver'):
    run(['git', 'clone', 'https://github.com/rodehyde/arc-agi-solver',
         '/content/arc-agi-solver'])
else:
    run(['git', '-C', '/content/arc-agi-solver', 'pull'])

# Dependencies
run([sys.executable, '-m', 'pip', 'install', '-q',
     'tiktoken', 'transformers', 'datasets'])

# HuggingFace authentication
# Add your token via: Tools → Secrets → Add new secret → Name: HF_TOKEN
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace: authenticated.')
except Exception as e:
    print(f'HuggingFace: no token found — public models only. ({e})')

print('Setup complete.')

In [ ]:
# ── Cell 4: Download ARC training data ──────────────────────────────────
import os, urllib.request, zipfile

ARC_DIR = '/content/arc-agi-solver/data/training'
os.makedirs(ARC_DIR, exist_ok=True)

if len(os.listdir(ARC_DIR)) < 400:
    print('Downloading ARC training data...')
    url = 'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    urllib.request.urlretrieve(url, '/tmp/arc.zip')
    with zipfile.ZipFile('/tmp/arc.zip') as z:
        for name in z.namelist():
            if 'data/training/' in name and name.endswith('.json'):
                task_name = name.split('/')[-1]
                with z.open(name) as src, open(f'{ARC_DIR}/{task_name}', 'wb') as dst:
                    dst.write(src.read())
    print(f'Downloaded {len(os.listdir(ARC_DIR))} tasks.')
else:
    print(f'{len(os.listdir(ARC_DIR))} tasks already present.')

In [ ]:
# ── Cell 5: Generate fine-tuning dataset (train.bin / val.bin) ──────────
import subprocess, sys

result = subprocess.run(
    [sys.executable,
     '/content/arc-agi-solver/scripts/prepare_arc_finetune.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# ── Cell 6: Set up nanoGPT data directory + config ─────────────────────
import os, shutil, subprocess, sys

# Create data/arc in nanoGPT and copy our bridge prepare.py
arc_data_dir = '/content/nanoGPT/data/arc'
os.makedirs(arc_data_dir, exist_ok=True)
shutil.copy('/content/arc-agi-solver/nanogpt/data/arc/prepare.py',
            f'{arc_data_dir}/prepare.py')

# Run prepare.py to copy train.bin / val.bin
os.chdir('/content/nanoGPT')
result = subprocess.run([sys.executable, 'data/arc/prepare.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

# Copy config and redirect checkpoints to Drive
shutil.copy('/content/arc-agi-solver/nanogpt/config/finetune_arc.py',
            '/content/nanoGPT/config/finetune_arc.py')
cfg = open('/content/nanoGPT/config/finetune_arc.py').read()
cfg = cfg.replace("out_dir = 'out-arc'",
                  f"out_dir = '{DRIVE_CKPT_DIR}'")
open('/content/nanoGPT/config/finetune_arc.py', 'w').write(cfg)
print(f'Config ready. Checkpoints → {DRIVE_CKPT_DIR}')

In [ ]:
# ── Cell 7: Train ───────────────────────────────────────────────────────
import subprocess, sys, os

os.chdir('/content/nanoGPT')
proc = subprocess.Popen(
    [sys.executable, '-u', 'train.py', 'config/finetune_arc.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 8: Evaluate Stage 1 — rule description quality ─────────────────
# Stage 1 only predicts the rule; no grid generation yet (that is Stage 2).
# Loads the checkpoint once, runs NUM_TASKS unique eval tasks, and shows
# the training context alongside MODEL PREDICTED RULE vs EXPECTED RULE.
import json, os, re, sys
import torch
import tiktoken

NUM_TASKS      = 5      # unique tasks to evaluate
MAX_NEW_TOKENS = 400    # enough for TYPE + RULE + STEPS + RELATIONSHIP
TEMPERATURE    = 0.8
TOP_K          = 50

# ── 1. Load model ─────────────────────────────────────────────────────────
sys.path.insert(0, '/content/nanoGPT')
from model import GPTConfig, GPT

ckpt_path = f'{DRIVE_CKPT_DIR}/ckpt.pt'
print(f'Loading checkpoint: {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location='cuda')
print(f"  saved at step {ckpt['iter_num']}, val loss {ckpt['best_val_loss']:.4f}")
gptconf = GPTConfig(**ckpt['model_args'])
model = GPT(gptconf)
sd = ckpt['model']
for k in list(sd):
    if k.startswith('_orig_mod.'):
        sd[k[len('_orig_mod.'):]] = sd.pop(k)
model.load_state_dict(sd)
model.eval()
model = model.to('cuda')
print(f'Ready ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)\n')

enc     = tiktoken.get_encoding('gpt2')
EOT_STR = '<|endoftext|>'

def predict(prompt_text):
    """Return the model's rule continuation up to EOT."""
    ids = enc.encode_ordinary(prompt_text)
    x   = torch.tensor(ids, dtype=torch.long, device='cuda')[None, ...]
    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            y = model.generate(x, MAX_NEW_TOKENS,
                               temperature=TEMPERATURE, top_k=TOP_K)
    full = enc.decode(y[0].tolist())
    cont = full[len(prompt_text):]
    if EOT_STR in cont:
        cont = cont[:cont.index(EOT_STR)]
    return cont.strip()

# ── 2. Load one example per unique task ──────────────────────────────────
eval_path = '/content/arc-agi-solver/data/finetune/eval.jsonl'
seen, examples = set(), []
with open(eval_path) as f:
    for line in f:
        ex = json.loads(line)
        if ex['task_id'] not in seen:
            seen.add(ex['task_id'])
            examples.append(ex)
        if len(examples) >= NUM_TASKS:
            break

# ── 3. Helpers ────────────────────────────────────────────────────────────
def parse_sections(text):
    sections = {}
    for p in text.split('\n\n'):
        if ':' in p:
            label, _, body = p.partition(':\n')
            sections[label.strip()] = body.strip()
    return sections

def show_grid(grid_text, indent='    ', max_rows=8):
    rows = grid_text.strip().splitlines()
    for r in rows[:max_rows]:
        print(indent + r)
    if len(rows) > max_rows:
        print(indent + f'... ({len(rows)} rows total)')

# ── 4. Run and display ───────────────────────────────────────────────────
for idx, ex in enumerate(examples):
    text  = ex['text']
    # Prompt ends at 'Rule: ' — model generates the rule from here
    split_pt = text.index('\n\nRule: ') + len('\n\nRule: ')
    prompt   = text[:split_pt]
    expected_rule = ex['rule']

    print(f'\n{"="*60}')
    print(f'Task {idx+1}/{NUM_TASKS}  {ex["task_id"]}  [{ex["category"]}]')
    print(f'{"="*60}')

    # Show training context
    sections = parse_sections(text)
    i = 1
    while f'Example {i} Input' in sections:
        print(f'\n  Train {i} Input:')
        show_grid(sections[f'Example {i} Input'])
        print(f'  Train {i} Output:')
        show_grid(sections[f'Example {i} Output'])
        i += 1
    if 'Test Input' in sections:
        print(f'\n  Test Input:')
        show_grid(sections['Test Input'])

    # Generate rule
    pred_rule = predict(prompt)
    truncated = len(enc.encode_ordinary(pred_rule)) >= MAX_NEW_TOKENS - 5

    print(f'\n  --- MODEL PREDICTED RULE ---')
    print(f'  {pred_rule or "(nothing generated)"}')
    if truncated:
        print(f'  (reached token limit — may be cut off)')

    print(f'\n  --- EXPECTED RULE ---')
    print(f'  {expected_rule}')
    print()

print('Done.')

In [ ]:
# ── Cell 9: Download checkpoint to local machine ────────────────────────
from google.colab import files
import os

ckpt_path = f'{DRIVE_CKPT_DIR}/ckpt.pt'
if os.path.exists(ckpt_path):
    files.download(ckpt_path)
else:
    print(f'Checkpoint not found at {ckpt_path}')